# Liquid Neural Networks (LNN) for ADS-B Anomaly Detection

This notebook demonstrates using Liquid Neural Networks for detecting anomalies in ADS-B aircraft data.

**Key Features:**
- Time-continuous dynamics with Neural ODEs
- Adaptive signal processing for irregular time-series
- Reconstruction-based anomaly detection

**Use Cases:**
- GNSS spoofing detection
- Signal quality monitoring
- Multi-sensor consistency checking

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add models to path
sys.path.append('..')

from lnn.liquid_neural_network import create_lnn_model, LNNAnomalyDetector
from data_utils import ADSBDataset, AnomalyDataset

# Set style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Load and Explore Data

In [ ]:
# Load ADS-B data
data_path = '../../golden_7day_eda_results/golden_7day_ml_dataset.csv'

if Path(data_path).exists():
    df = pd.read_csv(data_path)
    print(f"Loaded dataset: {len(df):,} records")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nFirst few rows:")
    display(df.head())
else:
    print(f"⚠️  Dataset not found at {data_path}")
    print("Please run the EDA script first to generate the ML dataset.")
    
    # Create dummy data for demonstration
    print("\nGenerating dummy data for demonstration...")
    n_samples = 5000
    df = pd.DataFrame({
        'timestamp': pd.date_range('2026-01-16', periods=n_samples, freq='1s'),
        'hex': np.random.choice(['ABC123', 'DEF456', 'GHI789'], n_samples),
        'lat': np.random.randn(n_samples) * 0.1 + 60.0,
        'lon': np.random.randn(n_samples) * 0.1 + 25.0,
        'alt_baro': np.random.randn(n_samples) * 1000 + 30000,
        'gs': np.random.randn(n_samples) * 50 + 450,
        'track': np.random.rand(n_samples) * 360,
        'rssi': np.random.randn(n_samples) * 5 - 30,
        'distance_km': np.random.rand(n_samples) * 100,
        'signal_deviation': np.random.randn(n_samples) * 2,
        'hour_sin': np.sin(2 * np.pi * np.arange(n_samples) / 24),
        'hour_cos': np.cos(2 * np.pi * np.arange(n_samples) / 24),
    })

## 3. Create Dataset and DataLoader

In [ ]:
from torch.utils.data import DataLoader

# Create anomaly detection dataset
dataset = AnomalyDataset(
    df,
    sequence_length=20,
    normalize=True
)

print(f"Dataset created: {len(dataset)} sequences")
print(f"Features: {dataset.features}")
print(f"Input size: {len(dataset.features)}")

# Create data loader
data_loader = DataLoader(dataset, batch_size=32, shuffle=False)

# Get a sample
x_sample, labels_sample = next(iter(data_loader))
print(f"\nSample batch:")
print(f"  Input shape: {x_sample.shape}")
print(f"  Labels shape: {labels_sample.shape}")

## 4. Create LNN Model

In [ ]:
# Model parameters
input_size = len(dataset.features)
hidden_size = 32
latent_size = 16

# Create model
model = create_lnn_model(
    input_size=input_size,
    model_type='anomaly_detector',
    hidden_size=hidden_size,
    latent_size=latent_size,
    num_layers=2
).to(device)

print("LNN Anomaly Detector created:")
print(f"  Input size: {input_size}")
print(f"  Hidden size: {hidden_size}")
print(f"  Latent size: {latent_size}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Test Forward Pass

In [ ]:
# Test model on sample data
model.eval()
with torch.no_grad():
    x_test = x_sample[:4].to(device)  # Take 4 samples
    
    reconstruction, anomaly_scores, latent = model(x_test)
    
    print("Forward pass successful!")
    print(f"  Input shape: {x_test.shape}")
    print(f"  Reconstruction shape: {reconstruction.shape}")
    print(f"  Anomaly scores shape: {anomaly_scores.shape}")
    print(f"  Latent shape: {latent.shape}")
    print(f"\nSample anomaly scores: {anomaly_scores[0, :5, 0].cpu().numpy()}")

## 6. Visualize Model Behavior

In [ ]:
# Detect anomalies
is_anomaly, combined_scores = model.detect_anomalies(x_test, threshold=0.5)

# Plot results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i in range(4):
    ax = axes[i // 2, i % 2]
    
    # Plot input vs reconstruction for first feature
    time_steps = range(x_test.shape[1])
    ax.plot(time_steps, x_test[i, :, 0].cpu(), 'b-', label='Input', linewidth=2)
    ax.plot(time_steps, reconstruction[i, :, 0].cpu(), 'r--', label='Reconstruction', linewidth=2)
    
    # Highlight anomalies
    anomaly_mask = is_anomaly[i, :, 0].cpu().numpy() > 0.5
    if anomaly_mask.any():
        ax.scatter(np.where(anomaly_mask)[0], 
                  x_test[i, anomaly_mask, 0].cpu(), 
                  c='red', s=100, marker='x', label='Anomaly', zorder=5)
    
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Feature Value')
    ax.set_title(f'Sample {i+1}: Input vs Reconstruction')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAnomalies detected: {is_anomaly.sum().item()} / {is_anomaly.numel()}")

## 7. Training (Quick Demo)

In [ ]:
# Quick training demo (5 epochs)
model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion_recon = nn.MSELoss()
criterion_anomaly = nn.BCELoss()

num_epochs = 5
losses = []

print("Training for 5 epochs (demo)...")
for epoch in range(num_epochs):
    epoch_loss = 0
    for x, labels in data_loader:
        x = x.to(device)
        labels = labels.to(device).unsqueeze(-1)
        
        optimizer.zero_grad()
        reconstruction, anomaly_scores, _ = model(x)
        
        recon_loss = criterion_recon(reconstruction, x)
        anomaly_loss = criterion_anomaly(anomaly_scores, labels)
        loss = recon_loss + 0.5 * anomaly_loss
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(data_loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{num_epochs}: Loss = {avg_loss:.4f}")

# Plot training curve
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs+1), losses, 'b-o', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('LNN Training Progress (Demo)')
plt.grid(True, alpha=0.3)
plt.show()

print("\n✅ Training demo complete!")

## 8. Summary and Next Steps

### What We've Done:
1. ✅ Loaded ADS-B data and created sequences
2. ✅ Built LNN anomaly detector with continuous-time dynamics
3. ✅ Tested forward pass and anomaly detection
4. ✅ Visualized reconstructions and anomalies
5. ✅ Demonstrated quick training loop

### For Full Training:

```bash
python analysis/models/lnn/train_lnn_anomaly_detector.py \
    --data_path analysis/golden_7day_eda_results/golden_7day_ml_dataset.csv \
    --epochs 100 \
    --hidden_size 64 \
    --latent_size 32
```

### Applications:
- **GNSS Spoofing Detection**: Identify anomalous signal patterns
- **Multi-Sensor Validation**: Cross-check sensor readings
- **Quality Monitoring**: Detect sensor degradation or interference
- **Ghost Aircraft**: Identify phantom tracks

### Key Advantages:
- ⚡ **Continuous-time**: Natural for irregular time-series
- 🔄 **Adaptive**: Learnable time constants adjust to data
- 🎯 **Robust**: Neural ODE provides smooth dynamics
- 📊 **Interpretable**: Reconstruction error shows where anomalies occur